# **Mountain Car Problem**

An under-powered car is stuck in a valley between two hills. The engine is **not strong enough** to drive straight up to the flag on the right hill, so the agent has to learn to rock back and forth, building up momentum, before it can reach the goal.

| | |
|---|---|
| **State** | continuous `[position, velocity]` |
| **Position** | `-1.2 → 0.6` (goal is at `position ≥ 0.5`) |
| **Velocity** | `-0.07 → 0.07` |
| **Actions** | `0 = push left`, `1 = no push`, `2 = push right` |
| **Reward** | `-1` on every step until the goal is reached |
| **Episode** | ends at the goal, or after 200 steps |

### Why not Dynamic Programming (like Frozen Lake)?
In Frozen Lake the state space was **discrete and small** (16 states) with a known transition table `env.P`, so we could sweep every state with value/policy iteration. Mountain Car has a **continuous** state space — there is no `env.P` to loop over. We solve it with **model-free, tabular Q-learning** by first *discretizing* the continuous `(position, velocity)` into a grid of bins.

## 1. Setup

In [ ]:
!pip install gymnasium

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

## 2. Explore the environment

In [ ]:
env = gym.make('MountainCar-v0')

print('Observation space:', env.observation_space)
print('  low  (pos, vel):', env.observation_space.low)
print('  high (pos, vel):', env.observation_space.high)
print('Action space     :', env.action_space, '  # 0=left, 1=none, 2=right')

state, info = env.reset()
print('Sample start state:', state)

## 3. Discretize the continuous state

We chop each of the two continuous dimensions into a fixed number of bins, turning the infinite state space into a finite grid we can store in a Q-table.

In [ ]:
n_bins = (20, 20)                       # bins for (position, velocity)
state_low  = env.observation_space.low
state_high = env.observation_space.high
bin_width  = (state_high - state_low) / np.array(n_bins)

def discretize(state):
    """Map a continuous [position, velocity] to integer grid indices."""
    idx = ((state - state_low) / bin_width).astype(int)
    idx = np.clip(idx, 0, np.array(n_bins) - 1)   # keep edge cases in range
    return tuple(idx)

print('Continuous', state, '->', 'bin', discretize(state))

## 4. The Q-table

Shape = `(position_bins, velocity_bins, n_actions)`. `Q[p, v, a]` estimates the expected return of taking action `a` in the discretized state `(p, v)`.

In [ ]:
q_table = np.zeros(n_bins + (env.action_space.n,))
print('Q-table shape:', q_table.shape)

## 5. Train with Q-learning

Update rule (off-policy temporal-difference control):

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma\,\max_{a'} Q(s',a') - Q(s,a)\,\big]$$

We use an **ε-greedy** policy that explores a lot early on and gradually exploits the learned values as `epsilon` decays.

In [ ]:
alpha         = 0.1      # learning rate
gamma         = 0.99     # discount factor
epsilon       = 1.0      # initial exploration
epsilon_min   = 0.01
epsilon_decay = 0.9995
episodes      = 5000
max_steps     = 200

rewards_per_episode = []

for episode in range(episodes):
    state, _ = env.reset()
    state = discretize(state)
    total_reward = 0

    for step in range(max_steps):
        # epsilon-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        next_state_d = discretize(next_state)

        # Q-learning update
        best_next = np.max(q_table[next_state_d])
        td_target = reward + gamma * best_next
        q_table[state][action] += alpha * (td_target - q_table[state][action])

        state = next_state_d
        total_reward += reward
        if terminated or truncated:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)

    if (episode + 1) % 500 == 0:
        avg = np.mean(rewards_per_episode[-500:])
        print(f'Episode {episode + 1:>4}/{episodes} | avg reward (last 500): {avg:7.1f} | epsilon: {epsilon:.3f}')

print('\nTraining done.')

## 6. Learning curve

Reward is `-1` per step, so a value closer to `0` (e.g. `-120`) means the car reached the flag faster. A flat `-200` means it never made it that episode.

In [ ]:
window = 100
moving_avg = np.convolve(rewards_per_episode, np.ones(window) / window, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(rewards_per_episode, alpha=0.3, label='reward per episode')
plt.plot(range(window - 1, len(rewards_per_episode)), moving_avg, color='red', label=f'{window}-episode moving average')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('Mountain Car — Q-learning training progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Evaluate the greedy policy

We turn off exploration (always pick `argmax` Q) and run 100 episodes to measure how reliably the trained agent reaches the goal.

In [ ]:
def run_greedy_episode(env, q_table):
    state, _ = env.reset()
    state = discretize(state)
    total_reward = 0
    for step in range(max_steps):
        action = np.argmax(q_table[state])
        next_state, reward, terminated, truncated, _ = env.step(action)
        state = discretize(next_state)
        total_reward += reward
        if terminated or truncated:
            return total_reward, terminated, step + 1
    return total_reward, False, max_steps

n_eval = 100
results = [run_greedy_episode(env, q_table) for _ in range(n_eval)]
successes = [r for r in results if r[1]]

print(f'Success rate : {len(successes)}/{n_eval} reached the flag')
print(f'Avg reward   : {np.mean([r[0] for r in results]):.1f}')
if successes:
    print(f'Avg steps to goal (successful runs): {np.mean([r[2] for r in successes]):.1f}')

## 8. Visualize the learned policy

For every `(position, velocity)` cell, which action does the agent prefer? You should see structure: push **right** when already moving right, push **left** when moving left — i.e. the agent learned to *pump* the swing to build momentum.

In [ ]:
policy = np.argmax(q_table, axis=2)   # 0=left, 1=none, 2=right

plt.figure(figsize=(7, 6))
plt.imshow(policy.T, origin='lower', aspect='auto', cmap='coolwarm',
           extent=[state_low[0], state_high[0], state_low[1], state_high[1]])
cbar = plt.colorbar(ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['push left', 'no push', 'push right'])
plt.xlabel('Position')
plt.ylabel('Velocity')
plt.title('Learned greedy policy')
plt.show()

## 9. Watch a trained run (optional)

Records one greedy episode to an MP4 you can play back in the notebook. Run on a machine/Colab with video support.

In [ ]:
# !pip install "gymnasium[other]" moviepy   # uncomment if RecordVideo deps are missing
from gymnasium.wrappers import RecordVideo

video_env = RecordVideo(gym.make('MountainCar-v0', render_mode='rgb_array'),
                        video_folder='mountain_car_video', episode_trigger=lambda e: True)
state, _ = video_env.reset()
state = discretize(state)
done = False
while not done:
    action = np.argmax(q_table[state])
    obs, reward, terminated, truncated, _ = video_env.step(action)
    state = discretize(obs)
    done = terminated or truncated
video_env.close()
print('Saved a greedy rollout to ./mountain_car_video/')